In [9]:
!wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/refs/heads/main/minsearch.py

--2026-03-22 18:54:04--  https://raw.githubusercontent.com/alexeygrigorev/minsearch/refs/heads/main/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4404 (4.3K) [text/plain]
Saving to: ‘minsearch.py.2’

minsearch.py.2      100%[===================>]   4.30K  --.-KB/s    in 0s      

2026-03-22 18:54:04 (40.8 MB/s) - ‘minsearch.py.2’ saved [4404/4404]



In [10]:
import minsearch

/workspaces/llm-zoomcamp/01-intro/minsearch.py:10: UserWarning: Now minsearch is available at pypi. Run 'pip install minsearch' or 'uv add minsearch' to use it. Remove the downloaded file ('rm minsearch.py') and re-install it.
  warnings.warn(


In [11]:
import json

In [12]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [13]:
documents = []
for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)


In [14]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [15]:
index = minsearch.Index(
    text_fields = ["question", "text", "section"],
    keyword_fields = ["course"]
)

In [16]:
q = 'the course has already started, Can I still enroll'

In [17]:
index.fit(documents)

In [18]:
boost = {'question': 3.0, 'section': 0.5}
results = index.search(
    query=q,
    boost_dict = boost,
    filter_dict = {'course': 'data-engineering-zoomcamp'},
    num_results = 5
)

In [25]:
results

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 202

In [19]:
from openai import OpenAI

In [20]:
client = OpenAI()

In [21]:
response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=[{"role": "user", "content": q}]
)

In [22]:
prompt_template = """
You are a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database. 
Use only facts from the CONTEXT when answering the QUESTION.
If the CONTEXT doesn't contain the answer, output NONE.

QUESTION: {question}
CONTEXT:{context}

""".strip()

context = ""
for doc in results:
    context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

prompt = prompt_template.format(question=q, context=context).strip()

In [23]:
context

"section: General course-related questions\nquestion: Course - Can I still join the course after the start date?\nanswer: Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.\n\nsection: General course-related questions\nquestion: Course - Can I follow the course after it finishes?\nanswer: Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.\n\nsection: General course-related questions\nquestion: Course - When will the course start?\nanswer: The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course 

In [24]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}
    
    results = index.search(
        query=query,
        boost_dict = boost,
        filter_dict = {'course': 'data-engineering-zoomcamp'},
        num_results = 10
    )
    return results

In [25]:
def build_prompt(query, search_results):
    prompt_template = """
    You are a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database. 
    Use only facts from the CONTEXT when answering the QUESTION.
    
    QUESTION: {question}
    CONTEXT:{context}
    
    """.strip()
    
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [26]:
def llm(prompt):
    response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [27]:
query = 'how do I run kafka?'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)

    answer = llm(prompt)
    return answer

In [28]:
rag('the course has already started. Can I still enroll?')

"Yes, you can still enroll even if the course has already started. You are eligible to submit the homework assignments, but do keep in mind the deadlines for submitting the final projects. It's important not to delay everything until the last minute."

In [89]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [1]:
from elasticsearch import Elasticsearch

In [2]:
es_client = Elasticsearch('http://localhost:9200', headers={"Accept": "application/vnd.elasticsearch+json; compatible-with=8"})

In [3]:
print(es_client.info())

{'name': 'e7e6575e8602', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'STK64n4lSCSo1UC2FQE8Ww', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [5]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course_questions"

es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course_questions'})

In [6]:
from tqdm.auto import tqdm

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:

for doc in tqdm(documents):
    es_client.index(index=index_name, document = doc)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [00:05<00:00, 177.87it/s]


In [47]:
query = 'Do I need an access of GCP?'

In [40]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }
    response = es_client.search(index=index_name, body=search_query)
    result_docs = []
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    return result_docs

In [45]:
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)

    answer = llm(prompt)
    return answer

In [51]:
rag("I am getting this error:invalid reference format: repository name must be lowercase. How Can I solve it?")

'To handle type errors from BigQuery and Parquet data, the key issue often arises when there are missing values in columns like `DOlocationID` and `PUlocationID`. By default, Pandas may cast these columns as float data type which can lead to a mismatch with the schema expected in BigQuery. To resolve this, you should cast the columns to the appropriate data type before loading them into Google Cloud Storage (GCS). Use the `astype` method and cast these columns to `Int64` (different from `int64` and allows for missing values):\n\n```python\ndf["PUlocationID"] = df.PUlocationID.astype("Int64")\ndf["DOlocationID"] = df.DOlocationID.astype("Int64")\n```\n\nIt is recommended to define the data type of all columns in the transformation section of your ETL pipeline before loading the data to BigQuery to avoid such errors.'

In [52]:
rag("Where can I find the Terraform 1.1.3 Linux")

'You can find the Terraform 1.1.3 Linux (AMD 64) here: [https://releases.hashicorp.com/terraform/1.1.3/terraform_1.1.3_linux_amd64.zip](https://releases.hashicorp.com/terraform/1.1.3/terraform_1.1.3_linux_amd64.zip).'

In [53]:
rag("I am getting this error:invalid reference format: repository name must be lowercase. How Can I solve it?")

'To solve the error "invalid reference format: repository name must be lowercase" when using Docker, particularly when mounting volumes on Windows, consider the following adjustments:\n\n1. Move your data to a directory without spaces, for example, from “C:/Users/Alexey Grigorev/git/…” to “C:/git/…”.\n\n2. Replace the “-v” part in your Docker command with one of these options:\n   - `-v /c:/some/path/ny_taxi_postgres_data:/var/lib/postgresql/data`\n   - `-v //c:/some/path/ny_taxi_postgres_data:/var/lib/postgresql/data`\n   - `-v /c/some/path/ny_taxi_postgres_data:/var/lib/postgresql/data`\n   - `-v //c/some/path/ny_taxi_postgres_data:/var/lib/postgresql/data`\n   - `--volume //driveletter/path/ny_taxi_postgres_data/:/var/lib/postgresql/data`\n\n3. Consider using `winpty` at the beginning of your Docker command if you\'re executing it in a Windows context.\n\n4. Add quotes around the volume mapping:\n   - `-v "/c:/some/path/ny_taxi_postgres_data:/var/lib/postgresql/data"`\n   - `-v "//c

In [54]:
rag("I am getting this error: Bad int64 value: 0.0 error")

'The error "Bad int64 value: 0.0" occurs because some `ehail_fee` values are null, and casting them directly to integers results in this error. To solve this, use the `safe_cast` function from `dbt_utils` in your code to cast these values safely to integers, like this:\n\n```jinja\n{{ dbt_utils.safe_cast(\'ehail_fee\', api.Column.translate_type("integer"))}} as ehail_fee,\n```\n\nAlternatively, you can use `safe_cast` without relying on `dbt_utils`, like so:\n\n```sql\nsafe_cast(ehail_fee as integer)\n```\n\nThis method returns NULL instead of throwing an error for problematic values.'

In [55]:
rag("How to solve : Invalid JWT Token  on WSL")

'To solve the "Invalid JWT Token" issue on WSL, there might be a time desynchronization on your machine that affects the JWT computation. You can fix this by synchronizing your system time with the command:\n\n```bash\nsudo hwclock -s\n```\n\nThis command sets your system time correctly and should resolve the token issue.'